In [ ]:
import pandas as pd
import altair as alt

# Cargar archivo CSV
lolla = pd.read_csv(r"/content/Base de datos Lollapalooza 1(Base de datos 1) (2).csv", delimiter=",", encoding="utf-8", dtype= {"edicion": int})
lolla.head()

,edicion lolla,artista,genero musical 01,genero musical 02,genero musical 03,integrantes,nacionalidad,headliner,escenario,veces en lollapalooza
0,2011,The Killers,Rock alternativo,Post-punk revival,New wave,4.0,Estadounidense,Sí,Coca Cola Zero Stage,1.0
1,2011,Jane's Addiction,Rock,Hard Rock,Punk Rock,4.0,Estadounidense,Sí,Claro Stage,1.0
2,2011,Kanye West,Hip Hop,Rap,Pop,1.0,Estadounidense,Sí,Coca Cola Zero Stage,1.0
3,2011,Cypress Hill,Rap,Hip Hop,Rap Rock,4.0,Estadounidense,No,Claro Stage,1.0
4,2011,30 Secons To Mars,Rock,Rock alternativo,Metal progresivo,4.0,Estadounidense,No,Coca Cola Zero Stage,1.0


In [ ]:
lolla.shape

(1053, 10)

In [ ]:
lolla.columns

Index(['edicion lolla ', 'artista ', 'genero musical 01', 'genero musical 02',
       'genero musical 03', 'integrantes', 'nacionalidad', 'headliner',
       'escenario', 'veces en lollapalooza'],
      dtype='object')

In [ ]:
lolla.info

<bound method DataFrame.info of       edicion lolla                  artista        genero musical 01  \
0               2011              The Killers        Rock alternativo   
1               2011         Jane's Addiction                    Rock   
2               2011               Kanye West                 Hip Hop   
3               2011             Cypress Hill                    Rap    
4               2011        30 Secons To Mars                    Rock   
...              ...                      ...                     ...   
1048            2026       La caravana mágica                     Pop   
1049            2026            Niebla Niebla                Shoegaze   
1050            2026  Claudio Valenzuela Trio        Rock alternativo   
1051            2026              Mau & Ricky                Reguetón   
1052            2026               Quilapayún  Fusión latinoamericana   

      genero musical 02  genero musical 03  integrantes    nacionalidad  \
0     Post-punk revival           New wave          4.0  Estadounidense   
1             Hard Rock          Punk Rock          4.0  Estadounidense   
2                   Rap                Pop          1.0  Estadounidense   
3               Hip Hop           Rap Rock          4.0  Estadounidense   
4      Rock alternativo   Metal progresivo          4.0  Estadounidense   
...                 ...                ...          ...             ...   
1048             Reggae            Hip hop          3.0         Chilena   
1049           Dreampop                NaN          3.0         Chilena   
1050              Pop            Post-punk          3.0         Chilena   
1051         Pop latino            Cumbias          2.0      Venezolana   
1052      Música andina  Música folclórica         11.0         Chilena   

     headliner             escenario  veces en lollapalooza  
0           Sí  Coca Cola Zero Stage                    1.0  
1           Sí           Claro Stage                    1.0  
2           Sí  Coca Cola Zero Stage                    1.0  
3           No           Claro Stage                    1.0  
4           No  Coca Cola Zero Stage                    1.0  
...        ...                   ...                    ...  
1048        No           Lotus Stage                    1.0  
1049        No           Lotus Stage                    1.0  
1050        No           Lotus Stage                    1.0  
1051        No           Lotus Stage                    1.0  
1052        No           Lotus Stage                    2.0  

[1053 rows x 10 columns]>

In [ ]:
# Limpieza de nombres de columnas y strings
lolla.columns = lolla.columns.str.strip()

cols_texto = ['artista', 'genero musical 01', 'genero musical 02',
              'genero musical 03', 'nacionalidad', 'headliner', 'escenario']
for col in cols_texto:
    lolla[col] = lolla[col].astype(str).str.strip()

# Identificar artistas chilenos (incluye nacionalidades mixtas, ej. "Chilena-Argentina")
lolla['es_chileno'] = lolla['nacionalidad'].str.lower().str.contains('chilena')

print(f"Artistas chilenos identificados: {lolla['es_chileno'].sum()} de {len(lolla)}")
lolla[lolla['es_chileno']][['edicion lolla', 'artista', 'nacionalidad']].head()

Artistas chilenos identificados: 407 de 1053


,edicion lolla,artista,nacionalidad
18,2011,Los Bunkers,Chilena
19,2011,Chico Trujillo,Chilena
34,2011,Francisca Valenzuela,Chilena
35,2011,Anita Tijoux,Chilena
36,2011,Quique Neira,Chilena


In [ ]:
# 1. Presencia de artistas chilenos por edición
presencia = lolla.groupby('edicion lolla').agg(
    artistas_chilenos=('es_chileno', 'sum')
).reset_index()

# 2. Cantidad de géneros musicales: solo artistas chilenos, juntando las 3 columnas de género
chilenos = lolla[lolla['es_chileno']].copy()

generos_largo = chilenos.melt(
    id_vars=['edicion lolla', 'artista'],
    value_vars=['genero musical 01', 'genero musical 02', 'genero musical 03'],
    value_name='genero'
)
generos_largo = generos_largo[
    generos_largo['genero'].notna() &
    (~generos_largo['genero'].str.lower().isin(['nan', '']))
]

generos_por_edicion = generos_largo.groupby('edicion lolla')['genero'].nunique().reset_index()
generos_por_edicion.columns = ['edicion lolla', 'generos_unicos']

# 3. Unir ambas métricas en una sola tabla
datos = presencia.merge(generos_por_edicion, on='edicion lolla', how='left')
datos

,edicion lolla,artistas_chilenos,generos_unicos
0,2011,17,28
1,2012,15,26
2,2013,18,32
3,2014,21,35
4,2015,30,36
5,2016,18,33
6,2017,29,37
7,2018,43,55
8,2019,49,50
9,2022,41,34


In [ ]:
base = alt.Chart(datos).encode(
    x=alt.X('edicion lolla:O', title='Edición Lollapalooza')
)

linea_artistas = base.mark_line(point=True, color='#00af9a', strokeWidth=3).encode(
    y=alt.Y('artistas_chilenos:Q', title='N° de artistas chilenos', axis=alt.Axis(titleColor='#00af9a')),
    tooltip=[
        alt.Tooltip('edicion lolla:O', title='Edición'),
        alt.Tooltip('artistas_chilenos:Q', title='N° de artistas chilenos')
    ]
)

linea_generos = base.mark_line(point=True, color='#ffade3', strokeWidth=3, strokeDash=[4,4]).encode(
    y=alt.Y('generos_unicos:Q', title='CANTIDAD DE GÉNEROS MUSICALES', axis=alt.Axis(titleColor='#ffade3')),
    tooltip=[
        alt.Tooltip('edicion lolla:O', title='Edición'),
        alt.Tooltip('generos_unicos:Q', title='Cantidad de géneros musicales')
    ]
)

grafico = alt.layer(linea_artistas, linea_generos).resolve_scale(
    y='independent'
).properties(
    title='Evolución de la presencia de artistas chilenos y cantidad de géneros musicales en Lollapalooza',
    width=700,
    height=420
)

grafico

alt.LayerChart(...)

In [ ]:
grafico.save('lollapalooza_chilenos.html')